# LightGBM Experiment — IEEE-CIS Fraud Detection

## 0. Setup & Imports

In [ ]:
import subprocess
subprocess.run(['pip', 'install', 'mlflow', 'dagshub', 'optuna', '--quiet'], capture_output=True)

import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightgbm as lgb
import mlflow, mlflow.sklearn
import dagshub
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings('ignore')

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score
from sklearn.base import BaseEstimator, TransformerMixin

print('LightGBM version:', lgb.__version__)

In [ ]:
DAGSHUB_USERNAME = 'YOUR_DAGSHUB_USERNAME'
DAGSHUB_REPO     = 'YOUR_REPO_NAME'
EXPERIMENT_NAME  = 'LightGBM_Training'

dagshub.init(repo_owner=DAGSHUB_USERNAME, repo_name=DAGSHUB_REPO, mlflow=True)
mlflow.set_experiment(EXPERIMENT_NAME)

## 1. Data Loading

In [ ]:
BASE_PATH = '/kaggle/input/ieee-fraud-detection/'

train = pd.read_csv(BASE_PATH + 'train_transaction.csv').merge(
        pd.read_csv(BASE_PATH + 'train_identity.csv'), on='TransactionID', how='left')
test  = pd.read_csv(BASE_PATH + 'test_transaction.csv').merge(
        pd.read_csv(BASE_PATH + 'test_identity.csv'),  on='TransactionID', how='left')

print(f'Train: {train.shape} | Test: {test.shape}')

## 2. Cleaning

In [ ]:
with mlflow.start_run(run_name='LightGBM_Cleaning'):

    # Drop >90% missing
    missing_pct = train.isnull().mean()
    drop_cols = missing_pct[missing_pct > 0.9].index.tolist()
    train.drop(columns=drop_cols, inplace=True)
    test.drop(columns=[c for c in drop_cols if c in test.columns], inplace=True)

    # Drop near-constant (>99.5% same value)
    quasi_const = [c for c in train.columns
                   if train[c].value_counts(normalize=True, dropna=False).iloc[0] > 0.995]
    train.drop(columns=quasi_const, inplace=True)
    test.drop(columns=[c for c in quasi_const if c in test.columns], inplace=True)

    # Email domain grouping
    for col in ['P_emaildomain', 'R_emaildomain']:
        if col in train.columns:
            top = train[col].value_counts().nlargest(10).index
            train[col] = train[col].where(train[col].isin(top), 'other')
            test[col]  = test[col].where(test[col].isin(top), 'other')

    mlflow.log_param('dropped_high_missing', len(drop_cols))
    mlflow.log_param('dropped_quasi_constant', len(quasi_const))
    mlflow.log_metric('cols_after_cleaning', train.shape[1])
    print(f'After cleaning: {train.shape}')

## 3. Feature Engineering

In [ ]:
with mlflow.start_run(run_name='LightGBM_Feature_Engineering'):

    def feature_engineering(df, is_train=True, stats=None):
        df = df.copy()

        # Time features
        df['hour']         = (df['TransactionDT'] / 3600) % 24
        df['day_of_week']  = (df['TransactionDT'] / (3600 * 24)) % 7
        df['is_night']     = ((df['hour'] >= 22) | (df['hour'] <= 6)).astype(int)
        df['is_weekend']   = (df['day_of_week'] >= 5).astype(int)

        # Amount
        df['TransactionAmt_log']    = np.log1p(df['TransactionAmt'])
        df['TransactionAmt_cents']  = df['TransactionAmt'] - df['TransactionAmt'].astype(int)
        df['TransactionAmt_isround']= (df['TransactionAmt_cents'] == 0).astype(int)

        # Aggregation stats (fit on train, apply to test)
        agg_stats = {}
        for grp in ['card1', 'card2', 'addr1']:
            if grp not in df.columns:
                continue
            if is_train:
                mean_val = df.groupby(grp)['TransactionAmt'].mean()
                std_val  = df.groupby(grp)['TransactionAmt'].std()
                agg_stats[f'{grp}_mean'] = mean_val
                agg_stats[f'{grp}_std']  = std_val
            else:
                mean_val = stats[f'{grp}_mean']
                std_val  = stats[f'{grp}_std']
            df[f'{grp}_amt_mean'] = df[grp].map(mean_val)
            df[f'{grp}_amt_std']  = df[grp].map(std_val)
            df[f'{grp}_amt_z']    = (df['TransactionAmt'] - df[f'{grp}_amt_mean']) / (df[f'{grp}_amt_std'] + 1e-5)

        # Email match
        if 'P_emaildomain' in df.columns and 'R_emaildomain' in df.columns:
            df['email_match'] = (df['P_emaildomain'] == df['R_emaildomain']).astype(int)

        # Card combo
        if 'card4' in df.columns and 'card6' in df.columns:
            df['card_combo'] = df['card4'].astype(str) + '_' + df['card6'].astype(str)

        # Missing value count
        df['nan_count'] = df.isnull().sum(axis=1)

        return df, agg_stats

    train_fe, agg_stats = feature_engineering(train, is_train=True)
    test_fe,  _         = feature_engineering(test,  is_train=False, stats=agg_stats)

    TARGET   = 'isFraud'
    DROP_COLS= ['TransactionID', 'TransactionDT', TARGET]

    # LightGBM can handle categoricals natively — just mark them
    cat_cols = [c for c in train_fe.select_dtypes(include='object').columns
                if c not in DROP_COLS]
    for col in cat_cols:
        le = LabelEncoder()
        combined = pd.concat([train_fe[col], test_fe[col]]).astype(str)
        le.fit(combined)
        train_fe[col] = le.transform(train_fe[col].astype(str))
        test_fe[col]  = le.transform(test_fe[col].astype(str))

    mlflow.log_metric('features_after_fe', train_fe.shape[1])
    print(f'Shape after FE: {train_fe.shape}')

## 4. Feature Selection

In [ ]:
with mlflow.start_run(run_name='LightGBM_Feature_Selection'):

    feature_cols = [c for c in train_fe.columns if c not in DROP_COLS]
    X = train_fe[feature_cols].fillna(-999)
    y = train_fe[TARGET]

    # Quick LightGBM importance
    quick_lgb = lgb.LGBMClassifier(
        n_estimators=200, max_depth=6, learning_rate=0.1,
        num_leaves=63, random_state=42, device='gpu', n_jobs=-1
    )
    quick_lgb.fit(X, y)
    importances = pd.Series(quick_lgb.feature_importances_, index=feature_cols)

    # Keep top 150
    top_features = importances.nlargest(150).index.tolist()

    # Also remove zero-importance features
    zero_imp = importances[importances == 0].index.tolist()

    selected = [f for f in top_features if f not in zero_imp]

    # Plot
    fig, ax = plt.subplots(figsize=(10, 8))
    importances.nlargest(30).plot(kind='barh', ax=ax)
    ax.set_title('Top 30 LightGBM Feature Importances')
    plt.tight_layout()
    plt.savefig('lgbm_importance.png', dpi=100)
    mlflow.log_artifact('lgbm_importance.png')

    mlflow.log_param('fs_method', 'lgbm_importance_top150')
    mlflow.log_metric('features_before', len(feature_cols))
    mlflow.log_metric('zero_importance_features', len(zero_imp))
    mlflow.log_metric('features_selected', len(selected))

    X_sel = X[selected]
    X_test_sel = test_fe[selected].fillna(-999)
    print(f'Selected {len(selected)} features')

## 5. Training

### 5a. Underfitted Baseline

In [ ]:
with mlflow.start_run(run_name='LightGBM_Underfitted'):
    params_under = dict(
        n_estimators=30, num_leaves=8, max_depth=3,
        learning_rate=0.5, min_child_samples=500,
        random_state=42, device='gpu'
    )
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(lgb.LGBMClassifier(**params_under), X_sel, y,
                              cv=cv, scoring='roc_auc', n_jobs=-1)
    mlflow.log_params(params_under)
    mlflow.log_metric('cv_auc_mean', scores.mean())
    mlflow.log_metric('cv_auc_std', scores.std())
    mlflow.log_param('note', 'intentionally_underfitted_few_leaves')
    print(f'[UNDERFITTED] CV AUC: {scores.mean():.4f} ± {scores.std():.4f}')

### 5b. Overfitted Model

In [ ]:
with mlflow.start_run(run_name='LightGBM_Overfitted'):
    params_over = dict(
        n_estimators=3000, num_leaves=500, max_depth=-1,
        learning_rate=0.3, min_child_samples=1,
        subsample=1.0, colsample_bytree=1.0,
        reg_alpha=0, reg_lambda=0,
        random_state=42, device='gpu'
    )
    m = lgb.LGBMClassifier(**params_over)
    m.fit(X_sel, y)
    train_auc = roc_auc_score(y, m.predict_proba(X_sel)[:, 1])
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    cv_auc = cross_val_score(lgb.LGBMClassifier(**params_over), X_sel, y,
                              cv=cv, scoring='roc_auc').mean()
    mlflow.log_params(params_over)
    mlflow.log_metric('train_auc', train_auc)
    mlflow.log_metric('cv_auc', cv_auc)
    mlflow.log_metric('overfit_gap', train_auc - cv_auc)
    mlflow.log_param('note', 'intentionally_overfitted_huge_leaves_no_reg')
    print(f'[OVERFITTED] Train: {train_auc:.4f} | CV: {cv_auc:.4f} | Gap: {train_auc-cv_auc:.4f}')

### 5c. Optuna Hyperparameter Search

In [ ]:
def lgb_objective(trial):
    params = {
        'n_estimators':      trial.suggest_int('n_estimators', 300, 2000),
        'num_leaves':        trial.suggest_int('num_leaves', 31, 300),
        'max_depth':         trial.suggest_int('max_depth', 4, 12),
        'learning_rate':     trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'subsample':         trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 100),
        'reg_alpha':         trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
        'reg_lambda':        trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True),
        'scale_pos_weight':  trial.suggest_float('scale_pos_weight', 1.0, 5.0),
        'random_state':      42,
        'device':            'gpu',
        'n_jobs':            -1
    }
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(lgb.LGBMClassifier(**params), X_sel, y,
                              cv=cv, scoring='roc_auc', n_jobs=-1)
    return scores.mean()

study = optuna.create_study(direction='maximize')
study.optimize(lgb_objective, n_trials=30, show_progress_bar=True)
best_params = study.best_params
best_params.update({'random_state': 42, 'device': 'gpu'})
print(f'Best AUC: {study.best_value:.4f}')

### 5d. Final Cross-Validation + Pipeline

In [ ]:
with mlflow.start_run(run_name='LightGBM_Final_CV') as final_run:
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    oof = np.zeros(len(y))
    test_preds = np.zeros(len(X_test_sel))
    fold_aucs  = []

    for fold, (tr_idx, val_idx) in enumerate(cv.split(X_sel, y)):
        X_tr, X_val = X_sel.iloc[tr_idx], X_sel.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

        m = lgb.LGBMClassifier(**best_params)
        m.fit(X_tr, y_tr,
              eval_set=[(X_val, y_val)],
              callbacks=[lgb.early_stopping(50, verbose=False),
                         lgb.log_evaluation(-1)])

        val_pred  = m.predict_proba(X_val)[:, 1]
        oof[val_idx] = val_pred
        test_preds  += m.predict_proba(X_test_sel)[:, 1] / 5
        fold_auc     = roc_auc_score(y_val, val_pred)
        fold_aucs.append(fold_auc)
        print(f'  Fold {fold+1}: AUC = {fold_auc:.4f}')

    oof_auc = roc_auc_score(y, oof)
    mlflow.log_params(best_params)
    mlflow.log_metric('oof_auc', oof_auc)
    mlflow.log_metric('cv_auc_mean', np.mean(fold_aucs))
    mlflow.log_metric('cv_auc_std',  np.std(fold_aucs))
    for i, a in enumerate(fold_aucs):
        mlflow.log_metric(f'fold_{i+1}_auc', a)
    print(f'OOF AUC: {oof_auc:.4f}')


# Build Pipeline
class LGBMFraudPreprocessor(BaseEstimator, TransformerMixin):
    def __init__(self, selected_features=None):
        self.selected_features = selected_features
        self.label_encoders_   = {}
        self.cat_cols_         = []

    def fit(self, X, y=None):
        df = self._engineer(X.copy())
        self.cat_cols_ = df.select_dtypes(include='object').columns.tolist()
        for col in self.cat_cols_:
            le = LabelEncoder()
            le.fit(df[col].astype(str))
            self.label_encoders_[col] = le
        return self

    def transform(self, X):
        df = self._engineer(X.copy())
        for col in self.cat_cols_:
            if col in df.columns:
                le  = self.label_encoders_[col]
                vals = df[col].astype(str)
                vals = vals.where(vals.isin(le.classes_), le.classes_[0])
                df[col] = le.transform(vals)
        if self.selected_features:
            avail = [f for f in self.selected_features if f in df.columns]
            df = df[avail]
        return df.fillna(-999)

    def _engineer(self, df):
        df['hour']       = (df['TransactionDT'] / 3600) % 24
        df['day_of_week']= (df['TransactionDT'] / (3600*24)) % 7
        df['is_night']   = ((df['hour'] >= 22) | (df['hour'] <= 6)).astype(int)
        df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)
        df['TransactionAmt_log']   = np.log1p(df['TransactionAmt'])
        df['TransactionAmt_cents'] = df['TransactionAmt'] - df['TransactionAmt'].astype(int)
        df['nan_count']  = df.isnull().sum(axis=1)
        if 'P_emaildomain' in df.columns and 'R_emaildomain' in df.columns:
            df['email_match'] = (df['P_emaildomain'] == df['R_emaildomain']).astype(int)
        if 'card4' in df.columns and 'card6' in df.columns:
            df['card_combo'] = df['card4'].astype(str) + '_' + df['card6'].astype(str)
        return df


X_raw  = train.drop(columns=['isFraud', 'TransactionID'], errors='ignore')
y_raw  = train['isFraud']

lgbm_pipeline = Pipeline([
    ('preprocessor', LGBMFraudPreprocessor(selected_features=selected)),
    ('classifier',   lgb.LGBMClassifier(**best_params))
])
lgbm_pipeline.fit(X_raw, y_raw)

with mlflow.start_run(run_name='LightGBM_Pipeline_Registry'):
    mlflow.log_params(best_params)
    mlflow.log_metric('oof_auc', oof_auc)
    mlflow.sklearn.log_model(
        sk_model=lgbm_pipeline,
        artifact_path='lgbm_fraud_pipeline',
        registered_model_name='LightGBM_Fraud_Pipeline'
    )
    print('LightGBM pipeline registered!')

np.save('lgbm_test_preds.npy', test_preds)